# 05. LightGBM

LightGBM on the final features (no `day`, no amount/balance ratio, no `full_balance_attempt`), compared with the
logistic regression baseline. The learning curve is saved for the README.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.features import FINAL_FEATURES, FINAL_CONTINUOUS, FINAL_BINARY, CATEGORICAL
from src.split import make_masks
from src.models import make_lgbm, make_logreg, prepare_lgbm_frame
from src.evaluation import pick_threshold, metrics_at_threshold

In [ ]:
model_df = pd.read_parquet("../data/processed/model_df.parquet")
train_mask, val_mask, test_mask = make_masks(model_df)

y_train = model_df.loc[train_mask, "is_fraud"]
y_val = model_df.loc[val_mask, "is_fraud"]
y_test = model_df.loc[test_mask, "is_fraud"]

X_train = prepare_lgbm_frame(model_df.loc[train_mask], FINAL_FEATURES)
X_val = prepare_lgbm_frame(model_df.loc[val_mask], FINAL_FEATURES)
X_test = prepare_lgbm_frame(model_df.loc[test_mask], FINAL_FEATURES)

## Training settings

My first attempt used a large `scale_pos_weight` (about 535) and early stopping on validation. Early stopping kept
halting after a handful of trees and the validation curve jumped around, so the numbers could not be trusted.
The final settings (in `src/models.py`) use a small learning rate, larger leaves, no extra class weight and a fixed
number of trees. The learning curve below shows that training is now smooth.

In [ ]:
lgbm = make_lgbm()
lgbm.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=["train", "validation"],
    eval_metric="average_precision",
)

curve = lgbm.evals_result_

plt.plot(curve["train"]["average_precision"], label="train")
plt.plot(curve["validation"]["average_precision"], label="validation")
plt.xlabel("Number of trees")
plt.ylabel("PR-AUC (LightGBM's own estimate)")
plt.legend()

Path("../report").mkdir(exist_ok=True)
plt.savefig("../report/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

LightGBM computes `average_precision` slightly differently from scikit-learn, so use the table below
(scikit-learn values) for reporting.

In [ ]:
rows = []

# LightGBM
p_val = lgbm.predict_proba(X_val)[:, 1]
p_test = lgbm.predict_proba(X_test)[:, 1]
t = pick_threshold(p_val, y_val)
for part, y, p in [("validation", y_val, p_val), ("test", y_test, p_test)]:
    m = metrics_at_threshold(y, p, t)
    m.update(model="LightGBM", part=part, threshold=round(t, 6))
    rows.append(m)

# Logistic regression without `day` (baseline from notebook 04)
cols_lr = FINAL_CONTINUOUS + ["log_amount_to_balance_ratio"] + FINAL_BINARY + CATEGORICAL
cont_lr = FINAL_CONTINUOUS + ["log_amount_to_balance_ratio"]
logreg = make_logreg(cont_lr, FINAL_BINARY, CATEGORICAL)
logreg.fit(model_df.loc[train_mask, cols_lr], y_train)
q_val = logreg.predict_proba(model_df.loc[val_mask, cols_lr])[:, 1]
q_test = logreg.predict_proba(model_df.loc[test_mask, cols_lr])[:, 1]
t_lr = pick_threshold(q_val, y_val)
for part, y, p in [("validation", y_val, q_val), ("test", y_test, q_test)]:
    m = metrics_at_threshold(y, p, t_lr)
    m.update(model="LogReg (no day)", part=part, threshold=round(t_lr, 6))
    rows.append(m)

results = pd.DataFrame(rows)
results[["model", "part", "threshold", "precision", "recall", "f1", "pr_auc", "TP", "FN", "FP"]].round(3)

## Notes

- Compare models by PR-AUC and by recall at a fixed recall level on validation. Precision depends on the fraud
  share in each set, so it should not be compared between validation and test.
- The large gap between LightGBM and logistic regression is investigated in the next notebook: it comes from
  the amount and balance features, not from a better understanding of fraud.